In [1]:
!pip install -q chromadb sentence-transformers rank_bm25

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 91.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/1

In [2]:
import chromadb
from chromadb.utils import embedding_functions

knowledge = [
    ("The AeroLite fan model is identified by code FAN-208. It uses a low-noise motor.", "catalog.txt", "product"),
    ("The HydroSense sensor has identifier SKU-7314 and measures water flow in a pipe.", "catalog.txt", "product"),
    ("Solar panels convert sunlight into electrical energy through photovoltaic cells.", "energy.txt", "renewable_energy"),
    ("A vector database stores embeddings and retrieves items by similarity.", "ai_notes.txt", "vector_search"),
    ("Hybrid search combines meaning-based retrieval with keyword matching.", "ai_notes.txt", "retrieval")
]

documents = [item[0] for item in knowledge]
metadatas = [
    {"source": item[1], "topic": item[2]}
    for item in knowledge
]
ids = [f"passage_{i}" for i in range(len(documents))]

client = chromadb.Client()
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
collection = client.get_or_create_collection(
    name="hybrid_search_practice",
    embedding_function=ef
)
collection.add(ids=ids, documents=documents, metadatas=metadatas)

print("Indexed passages:", collection.count())


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Indexed passages: 5


In [3]:
def semantic_search(query, n_results=4):
    result = collection.query(
        query_texts=[query],
        n_results=min(n_results, collection.count()),
        include=["documents", "metadatas", "distances"]
    )
    output = []
    for doc, meta, distance in zip(
        result["documents"][0],
        result["metadatas"][0],
        result["distances"][0]
    ):
        output.append({
            "text": doc, "source": meta["source"],
            "distance": float(distance)
        })
    return output

for item in semantic_search("Find product FAN-208", 3):
    print(round(item["distance"], 4), item["text"])


0.3635 The AeroLite fan model is identified by code FAN-208. It uses a low-noise motor.
0.8368 The HydroSense sensor has identifier SKU-7314 and measures water flow in a pipe.
0.8922 Solar panels convert sunlight into electrical energy through photovoltaic cells.


In [4]:
from rank_bm25 import BM25Okapi
import re

def tokenize(text):
    # Keep letter/number identifiers as tokens; split hyphens into searchable parts too.
    raw = re.findall(r"[a-zA-Z0-9]+", text.lower())
    return raw

tokenized_documents = [tokenize(doc) for doc in documents]
bm25 = BM25Okapi(tokenized_documents)

def bm25_search(query, n_results=4):
    scores = bm25.get_scores(tokenize(query))
    ranked_indices = sorted(
        range(len(scores)), key=lambda idx: scores[idx], reverse=True
    )
    output = []
    for idx in ranked_indices[:n_results]:
        if scores[idx] > 0:
            output.append({
                "text": documents[idx],
                "source": metadatas[idx]["source"],
                "bm25_score": float(scores[idx])
            })
    return output

print("BM25 results for exact code FAN-208:")
for item in bm25_search("FAN-208", 3):
    print(round(item["bm25_score"], 4), item["text"])


BM25 results for exact code FAN-208:
2.3553 The AeroLite fan model is identified by code FAN-208. It uses a low-noise motor.


In [5]:
meaning_query = "a quiet appliance that moves air"
print("Semantic search:")
for item in semantic_search(meaning_query, 2):
    print(item["text"])

print("\nBM25 search:")
for item in bm25_search(meaning_query, 2):
    print(item["text"])


Semantic search:
The AeroLite fan model is identified by code FAN-208. It uses a low-noise motor.
The HydroSense sensor has identifier SKU-7314 and measures water flow in a pipe.

BM25 search:
A vector database stores embeddings and retrieves items by similarity.
The HydroSense sensor has identifier SKU-7314 and measures water flow in a pipe.


In [6]:
def hybrid_search(query, n_results=4, rrf_k=60):
    semantic = semantic_search(query, n_results=len(documents))
    keyword = bm25_search(query, n_results=len(documents))

    scores = {}
    lookup = {}

    for rank, item in enumerate(semantic, start=1):
        key = item["text"]
        scores[key] = scores.get(key, 0.0) + 1 / (rrf_k + rank)
        lookup[key] = {"text": key, "source": item["source"]}

    for rank, item in enumerate(keyword, start=1):
        key = item["text"]
        scores[key] = scores.get(key, 0.0) + 1 / (rrf_k + rank)
        lookup[key] = {"text": key, "source": item["source"]}

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [
        {**lookup[key], "rrf_score": scores[key]}
        for key in ranked[:n_results]
    ]

for item in hybrid_search("What is the device with code SKU-7314?", 3):
    print(round(item["rrf_score"], 5), item["text"])


0.03279 The HydroSense sensor has identifier SKU-7314 and measures water flow in a pipe.
0.03226 The AeroLite fan model is identified by code FAN-208. It uses a low-noise motor.
0.0315 Hybrid search combines meaning-based retrieval with keyword matching.


In [7]:
import pandas as pd

test_query = "SKU-7314"
semantic_top = semantic_search(test_query, 1)
keyword_top = bm25_search(test_query, 1)
hybrid_top = hybrid_search(test_query, 1)

comparison = pd.DataFrame([
    {"Method": "Semantic only", "Top result": semantic_top[0]["text"] if semantic_top else "(none)"},
    {"Method": "BM25 keyword only", "Top result": keyword_top[0]["text"] if keyword_top else "(none)"},
    {"Method": "Hybrid RRF", "Top result": hybrid_top[0]["text"] if hybrid_top else "(none)"}
])
comparison


,Method,Top result
0,Semantic only,The HydroSense sensor has identifier SKU-7314 ...
1,BM25 keyword only,The HydroSense sensor has identifier SKU-7314 ...
2,Hybrid RRF,The HydroSense sensor has identifier SKU-7314 ...


In [8]:
def retrieve_with_threshold(query, n_results=4, min_similarity=0.30):
    matches = []
    for item in semantic_search(query, n_results):
        # This collection uses cosine distance by default with this embedding setup.
        similarity = 1 - item["distance"]
        if similarity >= min_similarity:
            matches.append({
                "text": item["text"],
                "source": item["source"],
                "similarity": similarity
            })
    return matches

print(retrieve_with_threshold("How do solar panels work?", min_similarity=0.30))


[{'text': 'Solar panels convert sunlight into electrical energy through photovoltaic cells.', 'source': 'energy.txt', 'similarity': 0.7678670287132263}]


In [9]:
KNOWN_TOPICS = sorted({meta["topic"] for meta in metadatas})

def smart_retrieve(query, n_results=4, min_similarity=0.30):
    semantic = retrieve_with_threshold(
        query, n_results=n_results, min_similarity=min_similarity
    )
    if semantic:
        return {
            "status": "semantic",
            "matches": semantic,
            "message": "Relevant passages found with semantic search."
        }

    hybrid = hybrid_search(query, n_results=n_results)
    # BM25/RRF can return weak matches, so only accept keyword fallback
    # when there is a positive BM25 score for at least one document.
    keyword_matches = bm25_search(query, n_results=n_results)
    if keyword_matches:
        return {
            "status": "hybrid_fallback",
            "matches": [
                {"text": item["text"], "source": item["source"]}
                for item in hybrid
            ],
            "message": "Semantic search found no confident match; hybrid fallback was used."
        }

    return {
        "status": "empty",
        "matches": [],
        "message": "No relevant passage was found. Available topics: " + ", ".join(KNOWN_TOPICS)
    }

print(smart_retrieve("What is the identifier of the HydroSense sensor?"))


{'status': 'semantic', 'matches': [{'text': 'The HydroSense sensor has identifier SKU-7314 and measures water flow in a pipe.', 'source': 'catalog.txt', 'similarity': 0.7349245548248291}], 'message': 'Relevant passages found with semantic search.'}


In [10]:
result = smart_retrieve("How do I bake sourdough bread?")
print("Status:", result["status"])
print(result["message"])
for match in result["matches"]:
    print("-", match["text"])


Status: empty
No relevant passage was found. Available topics: product, renewable_energy, retrieval, vector_search


In [11]:
def format_grounded_prompt(query, retrieval):
    if not retrieval["matches"]:
        return (
            "No answer should be generated from the knowledge base.\n"
            + retrieval["message"]
        )

    context = "\n\n".join(
        f"[Source: {item['source']}]\n{item['text']}"
        for item in retrieval["matches"]
    )
    return (
        "Answer using only the context below. If it does not contain the answer, "
        "say you do not know based on the provided documents.\n\n"
        f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    )

query = "What does SKU-7314 identify?"
retrieval = smart_retrieve(query)
print(format_grounded_prompt(query, retrieval))


Answer using only the context below. If it does not contain the answer, say you do not know based on the provided documents.

Context:
[Source: catalog.txt]
The HydroSense sensor has identifier SKU-7314 and measures water flow in a pipe.

Question: What does SKU-7314 identify?
Answer:
